In [1]:
import json
import pandas as pd

from pathlib import Path

In [2]:
with open("investigation_reports.json") as f:

    reports = json.load(f)

In [6]:
from collections import Counter


class BenchmarkEngine:

    def __init__(self, reports):

        self.reports = reports

    def generate(self):

        total = len(self.reports)

        unsafe = 0

        hallucination = 0

        guideline = 0

        supported = 0

        claims = 0

        failures = Counter()

        specialties = Counter()

        severities = Counter()

        for sample in self.reports:

            report = sample

            if report["safety"]["unsafe"]:

                unsafe += 1

            if report["hallucination"]:

                hallucination += 1

            if report["guideline"]["follows_guideline"]:

                guideline += 1

            for claim in report["claims"]:

                claims += 1

                if claim["supported"]:

                    supported += 1

            failures.update(

                report["failure_tags"]

            )

            severity = report["safety"]["severity"]

            severities[severity] += 1

            specialty = report["metadata"]["specialty"]

            if specialty:

                specialties[specialty] += 1

        return {

            "total_samples": total,

            "unsafe_rate":

                unsafe / total,

            "hallucination_rate":

                hallucination / total,

            "guideline_adherence":

                guideline / total,

            "claim_support_rate":

                supported / max(claims,1),

            "failure_distribution":

                dict(failures),

            "severity_distribution":

                dict(severities),

            "specialty_distribution":

                dict(specialties)

        }

In [5]:
print(reports[0].keys())

dict_keys(['metadata', 'prediction_summary', 'observations', 'failure_tags', 'inferences', 'claims', 'positive_findings', 'missing_information', 'safety', 'guideline', 'hallucination', 'failure_hypotheses', 'primary_failure', 'clinical_summary'])


In [7]:
benchmark = BenchmarkEngine(

    reports

).generate()

benchmark

{'total_samples': 1,
 'unsafe_rate': 0.0,
 'hallucination_rate': 0.0,
 'guideline_adherence': 0.0,
 'claim_support_rate': 0.0,
 'failure_distribution': {'Clinical Reasoning': 1,
  'Ethics': 1,
  'Communication': 1},
 'severity_distribution': {'None': 1},
 'specialty_distribution': {}}

In [8]:
print("="*70)

print("MEDALIGN BENCHMARK REPORT")

print("="*70)

print()

print(f"Samples               : {benchmark['total_samples']}")

print(f"Unsafe Rate           : {benchmark['unsafe_rate']:.2%}")

print(f"Hallucination Rate    : {benchmark['hallucination_rate']:.2%}")

print(f"Guideline Adherence   : {benchmark['guideline_adherence']:.2%}")

print(f"Claim Support Rate    : {benchmark['claim_support_rate']:.2%}")

print()

print("="*70)

print("Failure Distribution")

print("="*70)

for k,v in benchmark["failure_distribution"].items():

    print(f"{k:30} {v}")

print()

print("="*70)

print("Severity Distribution")

print("="*70)

for k,v in benchmark["severity_distribution"].items():

    print(f"{k:30} {v}")

print()

print("="*70)

print("Clinical Domains")

print("="*70)

for k,v in benchmark["specialty_distribution"].items():

    print(f"{k:30} {v}")

MEDALIGN BENCHMARK REPORT

Samples               : 1
Unsafe Rate           : 0.00%
Hallucination Rate    : 0.00%
Guideline Adherence   : 0.00%
Claim Support Rate    : 0.00%

Failure Distribution
Clinical Reasoning             1
Ethics                         1
Communication                  1

Severity Distribution
None                           1

Clinical Domains


In [9]:
class RecommendationEngine:

    def generate(

        self,

        benchmark

    ):

        recommendations = []

        if benchmark["unsafe_rate"] > 0.05:

            recommendations.append(

                "Increase safety-focused preference data."

            )

        if benchmark["hallucination_rate"] > 0.05:

            recommendations.append(

                "Collect hallucination correction examples."

            )

        failures = benchmark["failure_distribution"]

        if failures:

            top = max(

                failures,

                key=failures.get

            )

            recommendations.append(

                f"Collect additional training data for {top}."

            )

        return recommendations

In [10]:
recommendations = RecommendationEngine().generate(

    benchmark

)

print()

print("="*70)

print("Recommendations")

print("="*70)

for r in recommendations:

    print("-",r)


Recommendations
- Collect additional training data for Clinical Reasoning.


In [11]:
with open(

    "benchmark_report.json",

    "w"

) as f:

    json.dump(

        benchmark,

        f,

        indent=4

    )

print("Saved benchmark_report.json")

Saved benchmark_report.json
